In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("QVI_data.csv")
df.head()

,LYLTY_CARD_NBR,DATE,STORE_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES,PACK_SIZE,BRAND,LIFESTAGE,PREMIUM_CUSTOMER
0,1000,2018-10-17,1,1,5,Natural Chip Compny SeaSalt175g,2,6.0,175,NATURAL,YOUNG SINGLES/COUPLES,Premium
1,1002,2018-09-16,1,2,58,Red Rock Deli Chikn&Garlic Aioli 150g,1,2.7,150,RRD,YOUNG SINGLES/COUPLES,Mainstream
2,1003,2019-03-07,1,3,52,Grain Waves Sour Cream&Chives 210G,1,3.6,210,GRNWVES,YOUNG FAMILIES,Budget
3,1003,2019-03-08,1,4,106,Natural ChipCo Hony Soy Chckn175g,1,3.0,175,NATURAL,YOUNG FAMILIES,Budget
4,1004,2018-11-02,1,5,96,WW Original Stacked Chips 160g,1,1.9,160,WOOLWORTHS,OLDER SINGLES/COUPLES,Mainstream


In [3]:
df.shape

(264834, 12)

In [4]:
df.groupby("STORE_NBR")["LYLTY_CARD_NBR"].sum()

STORE_NBR
1         720905
2        1135736
3        4779490
4        7025622
5        6964664
         ...    
268    146463848
269    432483003
270    440291831
271    375501918
272    153249277
Name: LYLTY_CARD_NBR, Length: 272, dtype: int64

In [9]:
df[df["STORE_NBR"] == 77]

,LYLTY_CARD_NBR,DATE,STORE_NBR,TXN_ID,PROD_NBR,PROD_NAME,PROD_QTY,TOT_SALES,PACK_SIZE,BRAND,LIFESTAGE,PREMIUM_CUSTOMER
73365,77000,2019-03-28,77,74911,18,Cheetos Chs & Bacon Balls 190g,1,3.3,190,CHEETOS,MIDAGE SINGLES/COUPLES,Budget
73366,77000,2019-04-13,77,74912,69,Smiths Chip Thinly S/Cream&Onion 175g,1,3.0,175,SMITHS,MIDAGE SINGLES/COUPLES,Budget
73367,77000,2018-09-26,77,74910,36,Kettle Chilli 175g,2,10.8,175,KETTLE,MIDAGE SINGLES/COUPLES,Budget
73368,77001,2019-02-27,77,74913,7,Smiths Crinkle Original 330g,2,11.4,330,SMITHS,YOUNG FAMILIES,Mainstream
73369,77001,2019-01-21,77,74914,9,Kettle Tortilla ChpsBtroot&Ricotta 150g,2,9.2,150,KETTLE,YOUNG FAMILIES,Mainstream
...,...,...,...,...,...,...,...,...,...,...,...,...
264818,2330321,2018-07-30,77,236756,71,Twisties Cheese Burger 250g,2,8.6,250,TWISTIES,YOUNG SINGLES/COUPLES,Mainstream
264819,2330331,2018-11-18,77,236760,95,Sunbites Whlegrn Crisps Frch/Onin 90g,2,3.4,90,SUNBITES,RETIREES,Budget
264820,2330431,2018-07-31,77,236770,50,Tostitos Lightly Salted 175g,1,4.4,175,TOSTITOS,OLDER SINGLES/COUPLES,Mainstream
264821,2330461,2018-07-21,77,236777,87,Infuzions BBQ Rib Prawn Crackers 110g,1,3.8,110,INFUZIONS,OLDER FAMILIES,Budget


In [16]:
df["DATE"] = pd.to_datetime(df["DATE"])
df["MONTH"] = df["DATE"].dt.to_period("M")

# Monthly metrics for all stores
store_monthly = df.groupby(["STORE_NBR", "MONTH"]).agg({
    "TOT_SALES": "sum",
    "LYLTY_CARD_NBR": "nunique",
    "TXN_ID": "nunique"
}).reset_index()

# Avg transactions per customer
store_monthly["AVG_TXN_PER_CUSTOMER"] = (
    store_monthly["TXN_ID"] / store_monthly["LYLTY_CARD_NBR"]
)

In [30]:
def find_control_store(trial_store):
    trial_data = store_monthly[store_monthly["STORE_NBR"] == trial_store]
    
    correlations = []

    for store in store_monthly["STORE_NBR"].unique():
        if store != trial_store:
            control_data = store_monthly[store_monthly["STORE_NBR"] == store]
            
            merged = pd.merge(trial_data, control_data, on="MONTH", suffixes=("_trial", "_control"))
            
            if len(merged) > 0:
                corr = merged["TOT_SALES_trial"].corr(merged["TOT_SALES_control"])
                correlations.append((store, corr))

    control_df = pd.DataFrame(correlations, columns=["STORE_NBR", "CORRELATION"])
    control_df = control_df.sort_values(by="CORRELATION", ascending=False)
    
    return control_df.head(1)  # best match

In [31]:
df.columns

Index(['LYLTY_CARD_NBR', 'DATE', 'STORE_NBR', 'TXN_ID', 'PROD_NBR',
       'PROD_NAME', 'PROD_QTY', 'TOT_SALES', 'PACK_SIZE', 'BRAND', 'LIFESTAGE',
       'PREMIUM_CUSTOMER', 'MONTH'],
      dtype='object')

In [32]:
trial_stores = [77, 86, 88]

control_mapping = {}

for store in trial_stores:
    best_match = find_control_store(store)
    control_store = best_match.iloc[0]["STORE_NBR"]
    
    control_mapping[store] = control_store
    
    print(f"Trial Store {store} → Control Store {control_store}")

/Users/balihar/pyenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3057: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/balihar/pyenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2914: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/Users/balihar/pyenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2914: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/Users/balihar/pyenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3057: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/balihar/pyenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2914: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/Users/balihar/pyenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2914: RuntimeWarning: invalid value encounter

Trial Store 77 → Control Store 31.0
Trial Store 86 → Control Store 31.0
Trial Store 88 → Control Store 206.0


/Users/balihar/pyenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/balihar/pyenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/balihar/pyenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/balihar/pyenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/balihar/pyenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:3057: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/Users/balihar/pyenv/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2914: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, f

In [33]:
def compare_stores(trial_store, control_store):
    trial = store_monthly[store_monthly["STORE_NBR"] == trial_store]
    control = store_monthly[store_monthly["STORE_NBR"] == control_store]

    comparison = pd.merge(trial, control, on="MONTH", suffixes=("_trial", "_control"))

    # Differences
    comparison["sales_diff"] = comparison["TOT_SALES_trial"] - comparison["TOT_SALES_control"]
    comparison["customer_diff"] = comparison["LYLTY_CARD_NBR_trial"] - comparison["LYLTY_CARD_NBR_control"]
    comparison["txn_diff"] = (
        comparison["AVG_TXN_PER_CUSTOMER_trial"] - comparison["AVG_TXN_PER_CUSTOMER_control"]
    )

    return comparison

In [34]:
results = {}

for trial_store, control_store in control_mapping.items():
    comparison = compare_stores(trial_store, control_store)
    results[trial_store] = comparison
    
    print(f"\n=== Store {trial_store} vs Control {control_store} ===")
    print(comparison[["MONTH", "sales_diff", "customer_diff", "txn_diff"]].head())


=== Store 77 vs Control 31.0 ===
     MONTH  sales_diff  customer_diff  txn_diff
0  2018-09       219.2             41  0.047619
1  2018-11       236.5             40  0.073171

=== Store 86 vs Control 31.0 ===
     MONTH  sales_diff  customer_diff  txn_diff
0  2018-09       908.6            102  0.242718
1  2018-11       909.2             99  0.250000

=== Store 88 vs Control 206.0 ===
     MONTH  sales_diff  customer_diff  txn_diff
0  2018-07      1307.0            128  0.186047
1  2019-04      1434.8            127  0.265625
